In [10]:
# 1. Imports and Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, RandomizedSearchCV, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Optional: set working directory if needed
# import os
# os.chdir("C:\\Academic Projects\\DIABETES-DETECTION\\")

In [17]:
train = pd.read_csv('Diabetes-dataset_train_res.csv')
test = pd.read_csv('Diabetes-dataset_test.csv')

# Show duplicate columns, if any
duplicates = train.columns[train.columns.duplicated()].tolist()
print("Duplicate columns in train:", duplicates)

if duplicates:
    print("We need to rename these duplicates.")
    # Use pandas' internal method to deduplicate (adds .1, .2 etc.)
    train.columns = pd.io.parsers.ParserBase({'names': train.columns})._maybe_dedup_names(train.columns)

# Repeat for test set (though duplicates should be same)
duplicates_test = test.columns[test.columns.duplicated()].tolist()
if duplicates_test:
    test.columns = pd.io.parsers.ParserBase({'names': test.columns})._maybe_dedup_names(test.columns)

# Now verify no duplicates remain
assert not train.columns.duplicated().any(), "Still duplicates in train!"
assert not test.columns.duplicated().any(), "Still duplicates in test!"

X_train = train.drop('Outcome', axis=1)
y_train = train['Outcome']
X_test = test.drop('Outcome', axis=1)
y_test = test['Outcome']

print("All good – no duplicate columns.")

Duplicate columns in train: []
All good – no duplicate columns.


In [11]:
train = pd.read_csv('Diabetes-dataset_train_res.csv')
test = pd.read_csv('Diabetes-dataset_test.csv')

# Drop duplicate columns (keep the first occurrence)
train = train.loc[:, ~train.columns.duplicated()]
test = test.loc[:, ~test.columns.duplicated()]

X_train = train.drop('Outcome', axis=1)
y_train = train['Outcome']
X_test = test.drop('Outcome', axis=1)
y_test = test['Outcome']

# Optional: Verify no duplicates remain
assert not X_train.columns.duplicated().any(), "Still duplicate columns!"

In [18]:
# 3. Define and tune base models using RandomizedSearchCV
# (We'll keep the best estimators for later)

# Common cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=16)

# 3.1 Random Forest
param_rf = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}
rf = RandomForestClassifier(random_state=16)
rf_random = RandomizedSearchCV(rf, param_rf, n_iter=30, cv=cv, scoring='accuracy', random_state=16, n_jobs=-1)
rf_random.fit(X_train, y_train)
best_rf = rf_random.best_estimator_
print("Best RF params:", rf_random.best_params_)

# 3.2 XGBoost
param_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}
xgb = XGBClassifier(random_state=16, eval_metric='logloss')
xgb_random = RandomizedSearchCV(xgb, param_xgb, n_iter=20, cv=cv, scoring='accuracy', random_state=16, n_jobs=-1)
xgb_random.fit(X_train, y_train)
best_xgb = xgb_random.best_estimator_
print("Best XGB params:", xgb_random.best_params_)



Best RF params: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}
Best XGB params: {'subsample': 1.0, 'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.1, 'colsample_bytree': 0.8}


In [19]:
 #3.3 LightGBM
param_lgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, -1],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [31, 50, 70],
    'subsample': [0.8, 1.0]
}
lgb = LGBMClassifier(random_state=16, verbose=-1)
lgb_random = RandomizedSearchCV(lgb, param_lgb, n_iter=20, cv=cv, scoring='accuracy', random_state=16, n_jobs=-1)
lgb_random.fit(X_train, y_train)
best_lgb = lgb_random.best_estimator_
print("Best LGB params:", lgb_random.best_params_)

# 3.4 Logistic Regression
param_lr = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga']
}
lr = LogisticRegression(random_state=16, max_iter=1000)
lr_random = RandomizedSearchCV(lr, param_lr, n_iter=10, cv=cv, scoring='accuracy', random_state=16, n_jobs=-1)
lr_random.fit(X_train, y_train)
best_lr = lr_random.best_estimator_
print("Best LR params:", lr_random.best_params_)

# 3.5 SVM (with probability=True for later stacking)
param_svm = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1],
    'kernel': ['rbf', 'poly']
}
svm = SVC(probability=True, random_state=16)
svm_random = RandomizedSearchCV(svm, param_svm, n_iter=15, cv=cv, scoring='accuracy', random_state=16, n_jobs=-1)
svm_random.fit(X_train, y_train)
best_svm = svm_random.best_estimator_
print("Best SVM params:", svm_random.best_params_)

ValueError: 
All the 100 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
100 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Python313\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Python313\Lib\site-packages\lightgbm\sklearn.py", line 1560, in fit
    super().fit(
    ~~~~~~~~~~~^
        X,
        ^^
    ...<12 lines>...
        init_model=init_model,
        ^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "c:\Python313\Lib\site-packages\lightgbm\sklearn.py", line 1049, in fit
    self._Booster = train(
                    ~~~~~^
        params=params,
        ^^^^^^^^^^^^^^
    ...<6 lines>...
        

In [ ]:
# 4. Collect all tuned models and evaluate their cross-validation accuracy
models = {
    'rf': best_rf,
    'xgb': best_xgb,
    'lgb': best_lgb,
    'svm': best_svm,
    'lr': best_lr
}

cv_scores = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    cv_scores[name] = scores.mean()
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Sort models by CV accuracy
sorted_models = sorted(cv_scores.items(), key=lambda x: x[1], reverse=True)
print("\nTop models by CV accuracy:")
for name, score in sorted_models:
    print(f"{name}: {score:.4f}")

NameError: name 'best_lgb' is not defined

In [ ]:
# 5. Select top 4 models for stacking
top_names = [name for name, _ in sorted_models[:4]]
top_estimators = [(name, models[name]) for name in top_names]
print("Selected estimators for stacking:", top_names)

In [ ]:
# 6. Build a stacking ensemble with cross-validation
# Meta-model: we can use a tuned logistic regression
meta_lr = LogisticRegression(C=0.1, solver='liblinear', random_state=16)

stack = StackingClassifier(
    estimators=top_estimators,
    final_estimator=meta_lr,
    cv=5,                       # out-of-fold predictions
    stack_method='predict_proba'
)
stack.fit(X_train, y_train)

In [ ]:
# 7. Evaluate the stacking model on the test set
y_pred_stack = stack.predict(X_test)
acc_stack = accuracy_score(y_test, y_pred_stack)
print(f"Stacking Test Accuracy: {acc_stack:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_stack, target_names=['without diabetes', 'with diabetes']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_stack)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['without diabetes','with diabetes'],
            yticklabels=['without diabetes','with diabetes'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Stacking Ensemble')
plt.show()

In [ ]:
# 8. (Optional) Fine-tune the stacking ensemble with GridSearchCV
param_stack = {
    'final_estimator__C': [0.01, 0.1, 1, 10],
    'final_estimator__penalty': ['l1', 'l2'],
    'cv': [3, 5]   # number of folds for stacking
}
grid_stack = GridSearchCV(stack, param_stack, cv=5, scoring='accuracy')
grid_stack.fit(X_train, y_train)
best_stack = grid_stack.best_estimator_
print("Best stacking params:", grid_stack.best_params_)
print("Tuned stacking test accuracy:", accuracy_score(y_test, best_stack.predict(X_test)))

In [ ]:
# 9. Save the final model and the scaler (if needed for deployment)
import pickle

# Save the best stacking model
pickle.dump(best_stack, open('hybrid_model_enhanced.pkl', 'wb'))

# If you need the scaler that was used during transformation, load it (it should have been saved in the transformation notebook)
# scaler = pickle.load(open('scaler_enhanced.pkl', 'rb'))
# But for deployment, you'll need both the scaler and the model.

In [ ]:
# 10. (Optional) Compare with individual model performances on test set
for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name} test accuracy: {acc:.4f}")